# Lab 2: Self-Evaluation and Iteration Loop (20 minutes)

In this lab, you will build an intelligent agent that can evaluate its own translation quality and iteratively refine until it meets quality standards.

## Learning Objectives

**You'll Learn These Agentic Patterns:**
- ✅ Create custom evaluation tools: `@tool def evaluate_translation_quality()`
- ✅ Implement reflect-refine loops
- ✅ Agent self-assessment and autonomous improvement
- ✅ Quality-driven iteration logic


## Step 1: Setup and Import Tools

In [1]:
from strands import Agent
from strands.models import BedrockModel
import sys
sys.path.append('lab_helpers')
from translation_tools import evaluate_translation_quality, refine_translation_section
from utils import print_section_header, print_success, print_info

print_success("Translation tools imported!")
print_info("Ready to build self-evaluating agent")

✅ Translation tools imported!
ℹ️  Ready to build self-evaluating agent


## Step 2: Load Sample Document

In [2]:
# Load the same AWS Lambda documentation
with open('sample_data/aws_lambda_intro.txt', 'r', encoding='utf-8') as f:
    aws_doc_text = f.read()

# Use first paragraph for focused demo
paragraphs = aws_doc_text.split('\n\n')
first_paragraph = paragraphs[1] if len(paragraphs) > 1 else paragraphs[0]  # Skip title, get first real paragraph

print_section_header("Source Text")
print(first_paragraph)


  Source Text

AWS Lambda is a serverless compute service that lets you run code without provisioning or managing servers. Lambda runs your code on high-availability compute infrastructure and performs all the administration of the compute resources, including server and operating system maintenance, capacity provisioning and automatic scaling, and logging.


## Step 3: Understand the Evaluation Tool

**Core Pattern**: Tools that enable agent self-assessment

In [3]:
print_section_header("Evaluation Tool Overview")

print("""
🔧 evaluate_translation_quality Tool:

Purpose: Allows the agent to critique its own translation

Inputs:
  - source_text: Original English text
  - translated_text: Russian translation to evaluate
  - target_language: Target language name

Outputs:
  - quality_score: 0-100 (85+ is acceptable)
  - issues: List of specific problems found
  - needs_refinement: Boolean flag

Evaluation Criteria:
  1. Accuracy - Correct meaning preservation
  2. Fluency - Natural native-speaker quality
  3. Terminology - Technical term correctness
  4. Tone - Appropriate formality level

💡 Key Insight: The tool provides structure for the LLM to
   systematically evaluate its own output.
""")


  Evaluation Tool Overview


🔧 evaluate_translation_quality Tool:

Purpose: Allows the agent to critique its own translation

Inputs:
  - source_text: Original English text
  - translated_text: Russian translation to evaluate
  - target_language: Target language name

Outputs:
  - quality_score: 0-100 (85+ is acceptable)
  - issues: List of specific problems found
  - needs_refinement: Boolean flag

Evaluation Criteria:
  1. Accuracy - Correct meaning preservation
  2. Fluency - Natural native-speaker quality
  3. Terminology - Technical term correctness
  4. Tone - Appropriate formality level

💡 Key Insight: The tool provides structure for the LLM to
   systematically evaluate its own output.



## Step 4: Create Self-Evaluating Agent

**Core Strands Pattern**: `Agent(model=model, tools=[...], system_prompt="...")`

In [4]:
# Configure model
model = BedrockModel(
    model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0"
)

# Create agent with evaluation tools
iterative_translator = Agent(
    model=model,
    tools=[evaluate_translation_quality, refine_translation_section],
    system_prompt="""
    You are an intelligent translation agent with self-evaluation capabilities.
    
    Your workflow:
    1. Translate the text from English to Russian
    2. Use evaluate_translation_quality tool to assess your translation
    3. If quality_score < 85 or needs_refinement is true:
       - Identify specific issues
       - Use refine_translation_section to improve problematic parts
       - Re-evaluate the refined translation
    4. Repeat until quality_score >= 85
    5. Return final translation with iteration summary
    
    Translation Rules:
    - Keep AWS service names in English (Lambda, S3, DynamoDB, etc.)
    - Use natural, fluent Russian
    - Maintain technical accuracy
    - Use formal tone for technical documentation
    
    Show your reasoning:
    - Explain what issues you found
    - Describe how you refined the translation
    - Report final quality score
    """
)

print_success("Self-evaluating translation agent created!")
print_info("Agent can now iterate until quality threshold is met")

✅ Self-evaluating translation agent created!
ℹ️  Agent can now iterate until quality threshold is met


## Step 5: Run Iterative Translation

Watch the agent evaluate and refine its own work!

In [5]:
print_section_header("Iterative Translation with Self-Evaluation")

translation_request = f"""
Translate this AWS documentation to Russian and iteratively refine until quality is excellent:

{first_paragraph}

Use your evaluation tool to assess quality and refine as needed.
Show your iteration process.
"""

print("🤖 Agent is working...\n")
result = iterative_translator(translation_request)

print("\n" + "="*60)
print("📊 Final Result:")
print("="*60)
print(result.message)


  Iterative Translation with Self-Evaluation

🤖 Agent is working...

I'll translate this AWS documentation to Russian, evaluate it, and refine it as needed until the quality is excellent. Let me start with the initial translation.

### Initial Translation:

AWS Lambda - это бессерверная вычислительная служба, которая позволяет запускать код без подготовки или управления серверами. Lambda запускает ваш код на вычислительной инфраструктуре с высокой доступностью и выполняет всё администрирование вычислительных ресурсов, включая обслуживание серверов и операционной системы, подготовку мощностей, автоматическое масштабирование и ведение журналов.

Now let's evaluate the quality of this translation:
Tool #1: evaluate_translation_quality
Based on my evaluation, I've identified several issues with my initial translation. Let me refine these sections:
Tool #2: refine_translation_section

Tool #3: refine_translation_section

Tool #4: refine_translation_section
### Refined Translation:

AWS Lam

## Step 6: Analyze the Iteration Process

Let's examine what the agent did.

In [6]:
print_section_header("Agent Reasoning Analysis")

print("""
🔍 What to observe in the agent's output:

1. 🔄 Iteration Count:
   - How many refinement cycles did the agent perform?
   - Did it stop when quality threshold was met?

2. 🎯 Issues Identified:
   - What specific problems did the agent find?
   - Were they legitimate quality concerns?

3. ✨ Refinements Made:
   - How did the agent improve the translation?
   - Did the changes address the identified issues?

4. 📈 Quality Progression:
   - Did quality scores improve with each iteration?
   - What was the final quality score?

💡 Key Insight: The agent has agency to improve its own work
   autonomously, without human intervention.
""")

# Show conversation history to see tool calls
print("\n📝 Tool Usage Summary:")
print(f"Total messages in conversation: {len(iterative_translator.messages)}")
print("\nTip: Examine iterative_translator.messages to see detailed tool calls")


  Agent Reasoning Analysis


🔍 What to observe in the agent's output:

1. 🔄 Iteration Count:
   - How many refinement cycles did the agent perform?
   - Did it stop when quality threshold was met?

2. 🎯 Issues Identified:
   - What specific problems did the agent find?
   - Were they legitimate quality concerns?

3. ✨ Refinements Made:
   - How did the agent improve the translation?
   - Did the changes address the identified issues?

4. 📈 Quality Progression:
   - Did quality scores improve with each iteration?
   - What was the final quality score?

💡 Key Insight: The agent has agency to improve its own work
   autonomously, without human intervention.


📝 Tool Usage Summary:
Total messages in conversation: 18

Tip: Examine iterative_translator.messages to see detailed tool calls


## Step 7: Compare with Lab 1 Output

Let's see the improvement over one-shot translation.

In [7]:
print_section_header("Comparison: One-Shot vs Iterative")

# Create one-shot agent for comparison
oneshot_agent = Agent(
    model=model,
    system_prompt="""
    You are a professional technical translator.
    Translate from English to Russian.
    Keep AWS service names in English.
    Use natural, fluent Russian.
    Provide only the translation.
    """
)

oneshot_result = oneshot_agent(f"Translate to Russian:\n\n{first_paragraph}")

print("📌 One-Shot Translation (Lab 1 approach):")
print("-" * 60)
print(oneshot_result.message)
print("\n")

print("🔄 Iterative Translation (Lab 2 approach):")
print("-" * 60)
print("[See output from Step 5 above]")
print("\n")

print("🤔 Discussion Questions:")
print("1. Which translation sounds more natural?")
print("2. Which has better terminology consistency?")
print("3. Which approach would you trust for production use?")
print("4. What's the tradeoff? (Hint: time and cost)")


  Comparison: One-Shot vs Iterative

AWS Lambda — это бессерверная вычислительная служба, позволяющая запускать код без выделения и управления серверами. Lambda выполняет ваш код на вычислительной инфраструктуре с высокой доступностью и берет на себя все административные функции по управлению вычислительными ресурсами, включая обслуживание серверов и операционных систем, выделение мощностей, автоматическое масштабирование и ведение журналов.📌 One-Shot Translation (Lab 1 approach):
------------------------------------------------------------
{'role': 'assistant', 'content': [{'text': 'AWS Lambda — это бессерверная вычислительная служба, позволяющая запускать код без выделения и управления серверами. Lambda выполняет ваш код на вычислительной инфраструктуре с высокой доступностью и берет на себя все административные функции по управлению вычислительными ресурсами, включая обслуживание серверов и операционных систем, выделение мощностей, автоматическое масштабирование и ведение журналов.

## Step 8: Test with Full Document

Try the iterative approach on the complete AWS Lambda documentation.

In [8]:
print_section_header("Full Document Translation")

print("⚠️ Note: This will take longer as the agent processes more text")
print("and performs multiple evaluation cycles.\n")

# Optional: Uncomment to run full document translation
# full_translation_request = f"""
# Translate this complete AWS documentation to Russian with iterative refinement:
# 
# {aws_doc_text}
# 
# Use your evaluation tool and refine until quality >= 85.
# """
# 
# full_result = iterative_translator(full_translation_request)
# print(full_result.message)

print("💡 Exercise: Uncomment the code above to translate the full document")
print("   and observe how the agent handles longer text.")


  Full Document Translation

⚠️ Note: This will take longer as the agent processes more text
and performs multiple evaluation cycles.

💡 Exercise: Uncomment the code above to translate the full document
   and observe how the agent handles longer text.


## Lab 2 Complete - Key Takeaways

### 🎉 What You've Mastered:

✅ **Agentic Patterns:**
- `@tool` decorator for custom evaluation functions
- Reflect-refine loop implementation
- Agent self-assessment capabilities
- Quality-driven iteration logic

✅ **Key Improvements Over Lab 1:**
- Agent can evaluate its own output
- Autonomous quality improvement
- Transparent reasoning process
- Consistent quality threshold enforcement

✅ **What Makes This "Agentic":**
- Agent has agency to decide when work is complete
- Self-directed improvement without human intervention
- Tool use based on autonomous assessment
- Goal-oriented behavior (achieve quality >= 85)

**🎯 Remaining Gap:** Agent still lacks access to authoritative terminology sources.

### 🚀 Next: Lab 3 - Knowledge Base Integration

In Lab 3, you'll add:
- Bedrock Knowledge Base with AWS terminology
- Tool to query authoritative translations
- Terminology validation in the refinement loop
- Production-quality translation consistency
